**[NOTES — draft, for James to rewrite]**

This notebook cross-tabulates ED discharge disposition against the avoidable-visit algorithms (reviewer comment on disposition and the Resources algorithm). It previously had no markdown at all; a title cell with a TOC (Data / Check / Table 1) and short section headings are needed. Changes since the previous version:

- The cohort is the analysis sample as the main pipeline defines it: every eligible visit. The one-visit-per-patient restriction is gone (it lives in `R2_visits.ipynb` as the sensitivity analysis).
- A check cell locks the counts to the main pipeline (417,723 eligible; 209,429 flagged).
- Table 1: each disposition's row shows N (% of the analysis sample) and the share flagged by each algorithm — counts and percentages only, unchanged in method.
- Table 2 (disposition mix within each algorithm combination, patient-clustered CIs) has moved to `3_descriptive.ipynb`, where it is saved as Table S2.
- Glossary worth writing: the seven exclusive combinations plus "None", as labelled in `cell_labels`.

In [1]:
if(!require("pacman")) install.packages("pacman")
pacman::p_load(arrow, geepack, gt, knitr, tidyverse)

# cluster-aware CI helpers shared by the analysis notebooks
source("ci_helpers.R")


Loading required package: pacman



In [2]:
ed_data <- read_parquet(
    "../MIMICED/ed_join_cc.parquet",
    col_select = c(subject_id, stay_id, intime, acuity, age, race,
                   disposition, nyu, res, acuity_out)
  ) |>
  mutate(
    combined_group = paste0(nyu, res, acuity_out)
  )


In [3]:
eligible <- ed_data |>
  filter(!is.na(acuity) & !is.na(age) & !is.na(race))

analysis <- eligible |>
  mutate(
    disposition = factor(
      str_to_sentence(disposition),
      levels = c("Home", "Admitted", "Transfer", "Left without being seen",
                 "Eloped", "Left against medical advice", "Expired", "Other")
    )
  )

cell_labels <- c(
  "100" = "Diagnosis",
  "010" = "Resources",
  "001" = "Triage-Acuity",
  "110" = "Diagnosis + Resources",
  "011" = "Resources + Triage-Acuity",
  "101" = "Diagnosis + Triage-Acuity",
  "111" = "All three",
  "000" = "None"
)

analysis <- analysis |>
  mutate(combined_group = factor(cell_labels[combined_group], levels = cell_labels))

n <- list(
  analysis = nrow(analysis),
  flagged  = sum(analysis$combined_group != "None")
)


In [4]:
checks <- tibble(
  check    = c("Analysis sample", "Flagged by at least one algorithm"),
  observed = c(n$analysis, n$flagged),
  expected = c(417723L, 209429L)
) |>
  mutate(result = if_else(observed == expected, "PASS", "FAIL"))

checks |>
  print()

stopifnot(all(checks$result == "PASS"))

# A tibble: 2 × 4
  check                             observed expected result
  <chr>                                <int>    <int> <chr> 
1 Analysis sample                     417723   417723 PASS  
2 Flagged by at least one algorithm   209429   209429 PASS  


In [5]:
n_pct <- function(x, n) {
  sprintf("%s (%.1f%%)", format(x, big.mark = ","), 100 * x / n)
}

table_1 <- analysis |>
  group_by(Disposition = disposition) |>
  summarise(
    N = n(),
    Diagnosis       = n_pct(sum(nyu), N),
    Resources       = n_pct(sum(res), N),
    `Triage-Acuity` = n_pct(sum(acuity_out), N),
    `Any algorithm` = n_pct(sum(combined_group != "None"), N),
    .groups = "drop"
  ) |>
  mutate(N = n_pct(N, n$analysis))

table_1 |>
  kable(align = c("l", rep("r", 5)))

table_1 |>
  gt() |>
  gtsave(filename = "R4_disposition_table_1.docx")




|Disposition                 |               N|       Diagnosis|      Resources|  Triage-Acuity|   Any algorithm|
|:---------------------------|---------------:|---------------:|--------------:|--------------:|---------------:|
|Home                        | 240,087 (57.5%)| 114,384 (47.6%)| 80,998 (33.7%)| 27,247 (11.3%)| 167,371 (69.7%)|
|Admitted                    | 153,530 (36.8%)|  27,184 (17.7%)|       0 (0.0%)|     567 (0.4%)|  27,593 (18.0%)|
|Transfer                    |    6,962 (1.7%)|   1,188 (17.1%)|       0 (0.0%)|     105 (1.5%)|   1,257 (18.1%)|
|Left without being seen     |    6,064 (1.5%)|   4,280 (70.6%)|  5,930 (97.8%)|    899 (14.8%)|   6,017 (99.2%)|
|Eloped                      |    5,647 (1.4%)|   2,373 (42.0%)|  3,450 (61.1%)|     555 (9.8%)|   4,496 (79.6%)|
|Left against medical advice |    1,864 (0.4%)|     771 (41.4%)|    740 (39.7%)|      32 (1.7%)|   1,219 (65.4%)|
|Expired                     |      197 (0.0%)|      26 (13.2%)|       0 (0.0%)|      